# SimpleAgriChain: Sistem Ketertelusuran Rantai Pasok Pangan Organik
Notebook Demonstrasi & Pengujian Multi-Skenario Netral (Revisi 1)

---
## Ringkasan Konsep
SimpleAgriChain adalah implementasi blockchain permissioned internal tanpa konsensus Proof-of-Work (zero-mining) yang dirancang untuk menjamin integritas catatan rantai pasok pangan organik.

### Fitur Kunci & Batas Sistem:
1. **Kriptografi SHA-256 Kanonikal:** Setiap blok mengikat hash blok sebelumnya (`previous_hash`) menggunakan serialisasi JSON kanonikal (`sort_keys=True`).
2. **Penilaian Dua Sumber Risiko Pangan:**
   - **Hulu (Petani):** Ambang batas kadar air panen maksimal **14.0%**. Jika > 14.0%, beras berisiko mengalami jamur/penurunan mutu.
   - **Hilir (Distributor):** Ambang batas suhu transit maksimal **25.0°C**. Jika > 25.0°C, komoditas berisiko rusak karena penurunan pendinginan.
3. **Kejujuran Batas Sistem (Analogi Buku Besar Dilem Super):**
   - Sistem menjamin: catatan yang sudah masuk tidak dapat diubah tanpa terdeteksi (tamper-evident).
   - Sistem TIDAK menjamin: kebenaran data saat pertama kali diinput manual (garbage in, garbage out). Jika distributor menulis suhu 15°C dari awal, rantai tetap VALID.


## 1. Definisi Model Data & Engine Blockchain (Pure Python)
Implementasi modul inti `Block` dan `SimpleAgriChain` yang selaras dengan backend.


In [1]:
import hashlib
import json
from datetime import datetime, timezone
from typing import Dict, Any, List, Tuple, Optional

class Block:
    def __init__(self, index: int, batch_id: str, actor: str, data: Dict[str, Any], previous_hash: str, timestamp: Optional[str] = None, current_hash: Optional[str] = None):
        self.index = index
        self.timestamp = timestamp or datetime.now(timezone.utc).isoformat()
        self.batch_id = batch_id
        self.actor = actor
        self.data = data
        self.previous_hash = previous_hash
        self.current_hash = current_hash or self.compute_hash()

    def compute_hash(self) -> str:
        block_dict = {
            "index": self.index,
            "timestamp": self.timestamp,
            "batch_id": self.batch_id,
            "actor": self.actor,
            "data": self.data,
            "previous_hash": self.previous_hash
        }
        block_string = json.dumps(block_dict, sort_keys=True, separators=(',', ':'))
        return hashlib.sha256(block_string.encode('utf-8')).hexdigest()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "index": self.index,
            "timestamp": self.timestamp,
            "batch_id": self.batch_id,
            "actor": self.actor,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "current_hash": self.current_hash
        }

class SimpleAgriChain:
    def __init__(self):
        self.chain: List[Block] = []
        self.chain.append(self.create_genesis_block())

    def create_genesis_block(self) -> Block:
        return Block(
            index=0,
            batch_id="GENESIS",
            actor="SYSTEM",
            data={"message": "Genesis Block SimpleAgriChain", "version": "1.0.0"},
            previous_hash="0",
            timestamp="2026-11-08T10:00:00.000000Z"
        )

    def add_block(self, batch_id: str, actor: str, data: Dict[str, Any]) -> Block:
        previous_block = self.chain[-1]
        new_block = Block(
            index=len(self.chain),
            batch_id=batch_id,
            actor=actor,
            data=data,
            previous_hash=previous_block.current_hash,
            timestamp=datetime.now(timezone.utc).isoformat()
        )
        self.chain.append(new_block)
        return new_block

    def is_chain_valid(self) -> Tuple[bool, List[int]]:
        tampered = []
        if not self.chain:
            return False, []
        
        g = self.chain[0]
        if g.previous_hash != "0" or g.current_hash != g.compute_hash():
            tampered.append(0)

        for i in range(1, len(self.chain)):
            curr = self.chain[i]
            prev = self.chain[i - 1]
            if curr.previous_hash != prev.current_hash or curr.current_hash != curr.compute_hash():
                tampered.append(i)

        return len(tampered) == 0, tampered

    def trace_batch(self, batch_id: str) -> Optional[Dict[str, Any]]:
        blocks = [b for b in self.chain if b.batch_id == batch_id]
        if not blocks:
            return None
        timeline = []
        has_temp_alert = False
        max_temp = None
        has_moisture_alert = False
        max_moisture = None
        cert_valid = False
        no_cert = ""

        for step, b in enumerate(blocks, start=1):
            item = {
                "step": step,
                "index": b.index,
                "actor": b.actor,
                "timestamp": b.timestamp,
                "data": b.data,
                "current_hash": b.current_hash
            }
            if b.actor == "PETANI":
                cert = b.data.get("no_sertifikat", "")
                if cert:
                    cert_valid = True
                    no_cert = cert
                try:
                    ka = float(b.data.get("kadar_air", 0))
                    if max_moisture is None or ka > max_moisture:
                        max_moisture = ka
                    alert_ka = ka > 14.0
                    item["moisture_alert"] = alert_ka
                    if alert_ka:
                        has_moisture_alert = True
                except (ValueError, TypeError):
                    item["moisture_alert"] = False

            elif b.actor == "DISTRIBUTOR":
                try:
                    suhu = float(b.data.get("suhu", 0))
                    if max_temp is None or suhu > max_temp:
                        max_temp = suhu
                    alert = suhu > 25.0
                    item["temperature_alert"] = alert
                    if alert:
                        has_temp_alert = True
                except (ValueError, TypeError):
                    item["temperature_alert"] = False
            timeline.append(item)

        has_quality_risk = bool(has_temp_alert or has_moisture_alert)
        risk_sources = []
        if has_moisture_alert:
            risk_sources.append("PETANI")
        if has_temp_alert:
            risk_sources.append("DISTRIBUTOR")

        return {
            "batch_id": batch_id,
            "total_records": len(timeline),
            "has_temperature_alert": has_temp_alert,
            "max_temperature": max_temp,
            "temperature_threshold": 25.0,
            "has_moisture_alert": has_moisture_alert,
            "kadar_air_max": max_moisture,
            "moisture_threshold": 14.0,
            "has_quality_risk": has_quality_risk,
            "risk_sources": risk_sources,
            "certificate_valid": cert_valid,
            "no_sertifikat": no_cert,
            "timeline": timeline
        }

print('Blockchain engine SimpleAgriChain siap!')


Blockchain engine SimpleAgriChain siap!


## 2. Skenario 1: Alur Normal Rantai Pasok Pangan
Petani mencatat kadar air aman (12.5% <= 14.0%), Distributor mencatat suhu dingin aman (22.0°C <= 25.0°C).


In [2]:
agrichain = SimpleAgriChain()

block1 = agrichain.add_block(
    batch_id="ORG-BRS-001",
    actor="PETANI",
    data={
        "komoditas": "Beras Merah Organik",
        "lokasi_lahan": "Desa Sukatani, Karawang",
        "tanggal_panen": "2026-11-05",
        "pupuk": "Kompos Organik Fermentasi",
        "no_sertifikat": "SNI-ORGANIK-2026-0881",
        "kadar_air": 12.5
    }
)

block2 = agrichain.add_block(
    batch_id="ORG-BRS-001",
    actor="DISTRIBUTOR",
    data={
        "tanggal_kirim": "2026-11-06",
        "rute": "Karawang -> Jakarta Timur",
        "suhu": 22.0,
        "status_kondisi": "Suhu Dingin Terjaga Baik"
    }
)

is_valid, _ = agrichain.is_chain_valid()
trace1 = agrichain.trace_batch("ORG-BRS-001")
print(f"Status Integritas: {'VALID' if is_valid else 'INVALID'}")
print(f"Ada Risiko Mutu: {trace1['has_quality_risk']} | Sumber Risiko: {trace1['risk_sources']}")
assert is_valid == True
assert trace1["has_quality_risk"] == False


Status Integritas: VALID
Ada Risiko Mutu: False | Sumber Risiko: []


## 3. Skenario 2: Deteksi Risiko Mutu dari Tahap Panen Petani
Petani mencatat kadar air basah (18.5% > 14.0%), meskipun suhu distributor dingin (23.0°C).
Sistem mendeteksi sumber risiko hulu (`risk_sources: ['PETANI']`).


In [3]:
block3 = agrichain.add_block(
    batch_id="ORG-BASAH-002",
    actor="PETANI",
    data={
        "komoditas": "Beras Hitam Organik",
        "lokasi_lahan": "Karawang",
        "tanggal_panen": "2026-11-05",
        "pupuk": "Kompos",
        "no_sertifikat": "SNI-002",
        "kadar_air": 18.5
    }
)

block4 = agrichain.add_block(
    batch_id="ORG-BASAH-002",
    actor="DISTRIBUTOR",
    data={
        "tanggal_kirim": "2026-11-06",
        "rute": "Karawang -> Jakarta",
        "suhu": 23.0,
        "status_kondisi": "Transit Dingin"
    }
)

trace2 = agrichain.trace_batch("ORG-BASAH-002")
print(f"Batch: {trace2['batch_id']}")
print(f"Peringatan Kadar Air: {trace2['has_moisture_alert']} (Kadar: {trace2['kadar_air_max']}%)")
print(f"Peringatan Suhu: {trace2['has_temperature_alert']}")
print(f"Sumber Risiko Terdeteksi: {trace2['risk_sources']}")
assert trace2["has_moisture_alert"] == True
assert trace2["risk_sources"] == ["PETANI"]


Batch: ORG-BASAH-002
Peringatan Kadar Air: True (Kadar: 18.5%)
Peringatan Suhu: False
Sumber Risiko Terdeteksi: ['PETANI']


## 4. Skenario 3: Demonstrasi "Bohong Sejak Awal" (Batas Sistem)
Distributor menginput suhu 15.0°C sejak awal di form, meskipun suhu asli truk panas.
Karena data belum pernah diubah pasca-catat, rantai tetap **VALID**. Ini membuktikan sistem menjamin *tamper-evidence*, bukan kebenaran input awal.


In [4]:
block5 = agrichain.add_block(
    batch_id="ORG-BOHONG-003",
    actor="PETANI",
    data={
        "komoditas": "Beras Pandan Wangi",
        "lokasi_lahan": "Karawang",
        "tanggal_panen": "2026-11-05",
        "pupuk": "Kompos",
        "no_sertifikat": "SNI-003",
        "kadar_air": 12.0
    }
)

# Distributor mengetik 15.0 sejak awal
block6 = agrichain.add_block(
    batch_id="ORG-BOHONG-003",
    actor="DISTRIBUTOR",
    data={
        "tanggal_kirim": "2026-11-06",
        "rute": "Karawang -> Bandung",
        "suhu": 15.0,
        "status_kondisi": "Suhu Dingin Terjaga Baik (Diinput 15 sejak awal)"
    }
)

is_valid, tampered = agrichain.is_chain_valid()
print(f"Status Integritas Rantai: {'VALID' if is_valid else 'INVALID'} (Tampered: {tampered})")
print("Catatan: Rantai tetap VALID karena tidak ada data yang diubah setelah tercatat.")
assert is_valid == True


Status Integritas Rantai: VALID (Tampered: [])
Catatan: Rantai tetap VALID karena tidak ada data yang diubah setelah tercatat.


## 5. Skenario 4: Simulasi Serangan Manipulasi Data Historis
Oknum mengubah data pada Blok #4 secara paksa di ledger. Rantai langsung terdeteksi **INVALID**.


In [5]:
# Manipulasi data pada Blok #4
agrichain.chain[4].data['suhu'] = 16.0
agrichain.chain[4].data['status_kondisi'] = "Data Palsu Pasca Catat"

is_valid, tampered_indices = agrichain.is_chain_valid()
print(f"Status Integritas Rantai Setelah Manipulasi: {'VALID' if is_valid else 'INVALID'}")
print(f"Indeks Blok yang Rusak/Dimanipulasi: {tampered_indices}")
assert is_valid == False
assert 4 in tampered_indices


Status Integritas Rantai Setelah Manipulasi: INVALID
Indeks Blok yang Rusak/Dimanipulasi: [4]


## 6. Kesimpulan & Nilai Pembelajaran
1. **Blockchain adalah Append-Only Ledger:** Menjamin apa yang sudah masuk tidak dapat diganti diam-diam.
2. **Oracle Problem Terbuka:** Data input manual dari manusia (baik petani maupun distributor) bisa tidak jujur. Solusi dunia nyata membutuhkan sensor terotentikasi kriptografis.
3. **Penilaian Risiko Dua Arah:** Kualitas pangan diuji secara adil dari kadar air hulu maupun suhu rantai dingin hilir.
